In [1]:
import random
import re
import pandas as pd
import matplotlib.pyplot as plt
import arabic_reshaper
from matplotlib import rcParams
from matplotlib import font_manager
import seaborn as sns
import missingno as msno
import numpy as np
# from transformers import pipeline
# from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
# from textblob import TextBlob
# from google.cloud import language_v1
# from sklearn.preprocessing import LabelEncoder
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import KMeans
# from sklearn.decomposition import PCA
# from sklearn.metrics import silhouette_score
# from scipy.spatial.distance import cdist
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# import torch
import pandas as pd


In [16]:
from Kobo.kobo_toolbox_client import KoboToolboxClient

# Initialize client
kobo_client = KoboToolboxClient(
	url='https://kf.kobotoolbox.org/',
	token='f7196ace51297457806144859ca0991c1747c11c'
)

In [19]:
try:
    forms = kobo_client.get_all_forms()
    if not forms:
        exit()
    
    # Get details of first form
    first_form_uid = forms[0].uid
    form_details = kobo_client.get_form_details(first_form_uid)
    
    # Get submissions (basic)
    submissions= kobo_client.get_submissions(first_form_uid)
    
    display(submissions.head())
    
except Exception as e:
    print(f'\nError occurred: {e}')

,gender,age_group,governorate,education_level,employment_status,specialization,satisfaction_overall,satisfaction_access,satisfaction_support,suggestions,...,_uuid,rootUuid,_attachments,_status,_submission_time,_tags,_notes,_validation_status,_submitted_by,_index
0,option_0,option_1,option_0,option_2,option_0,option_0,option_1,option_2,option_1,option_1,...,00239809-cd17-4a06-ac10-fb5fc875aaa6,uuid:00239809-cd17-4a06-ac10-fb5fc875aaa6,[],submitted_via_web,2025-07-09T05:52:30,[],[],{},None,1
1,option_0,option_1,option_0,option_2,option_0,option_0,option_1,option_2,option_1,option_1,...,72475b88-fbad-41fb-97db-ac859162db1b,uuid:72475b88-fbad-41fb-97db-ac859162db1b,[],submitted_via_web,2025-07-09T05:53:11,[],[],{},None,2
2,option_1,option_2,option_3,option_3,option_1,option_3,option_2,option_3,option_2,option_0,...,c80a7d33-629a-481b-a9ca-0a4a2be57a64,uuid:c80a7d33-629a-481b-a9ca-0a4a2be57a64,[],submitted_via_web,2025-07-09T05:53:45,[],[],{},None,3
3,option_0,option_1,option_0,option_2,option_1,option_1,option_3,option_2,option_1,option_0,...,fe4810d4-5b51-4aa2-b115-fd744b61a42f,uuid:fe4810d4-5b51-4aa2-b115-fd744b61a42f,[],submitted_via_web,2025-07-09T05:54:09,[],[],{},None,4


In [4]:
from Kobo.kobo_form_builder import KoboFormBuilder

builder = KoboFormBuilder('استبيان رضا المستفيدين')

# ------------------------
# القسم 1: الأسئلة الديموغرافية (الكل اختيارات)
# ------------------------

builder.add_select_question(
    name='gender',
    label='ما هو جنسك؟',
    options=['ذكر', 'أنثى'],
    required=True
)

builder.add_select_question(
    name='age_group',
    label='ما هو فئتك العمرية؟',
    options=['أقل من 18', '18-24', '25-34', '35-44', '45 فأكثر'],
    required=True
)

builder.add_select_question(
    name='governorate',
    label='في أي محافظة تعيش؟',
    options=['دمشق', 'ريف دمشق', 'حلب', 'حمص', 'حماة', 'اللاذقية', 'طرطوس', 'إدلب', 'درعا', 'القنيطرة', 'دير الزور', 'الحسكة', 'الرقة', 'السويداء'],
    required=True
)

builder.add_select_question(
    name='education_level',
    label='ما هو مستواك التعليمي؟',
    options=['ثانوي أو أقل', 'دبلوم', 'بكالوريوس', 'ماجستير', 'دكتوراه'],
    required=True
)

builder.add_select_question(
    name='employment_status',
    label='ما هو وضعك الوظيفي؟',
    options=['طالب', 'موظف', 'باحث عن عمل', 'عمل حر', 'أخرى'],
    required=True
)

builder.add_select_question(
    name='specialization',
    label='ما هو تخصصك الأكاديمي؟',
    options=['هندسة حاسوب', 'ذكاء اصطناعي', 'إدارة أعمال', 'اقتصاد', 'علوم اجتماعية', 'تربية', 'أخرى'],
    required=False
)

# ------------------------
# القسم 2: قياس رضا المستفيدين
# ------------------------

builder.add_select_question(
    name='satisfaction_overall',
    label='كيف تقيم تجربتك العامة مع الخدمات التي حصلت عليها؟',
    options=['راضٍ جدًا', 'راضٍ', 'محايد', 'غير راضٍ', 'غير راضٍ أبدًا'],
    required=True
)

builder.add_select_question(
    name='satisfaction_access',
    label='ما مدى سهولة الوصول إلى الخدمات المقدمة؟',
    options=['سهل جدًا', 'سهل', 'متوسط', 'صعب', 'صعب جدًا'],
    required=True
)

builder.add_select_question(
    name='satisfaction_support',
    label='ما مدى جودة الدعم الذي حصلت عليه أثناء استخدام الخدمة؟',
    options=['ممتاز', 'جيد جدًا', 'جيد', 'ضعيف', 'سيء جدًا'],
    required=True
)

builder.add_select_question(
    name='suggestions',
    label='هل لديك اقتراحات لتحسين الخدمات؟',
    options=['نعم، لدي اقتراح', 'لا يوجد لدي اقتراح'],
    required=False
)

# ------------------------
# Export as JSON
# ------------------------

print(builder.to_json())


{
  "survey": [
    {
      "type": "select_one",
      "name": "gender",
      "label": [
        "\u0645\u0627 \u0647\u0648 \u062c\u0646\u0633\u0643\u061f"
      ],
      "required": true,
      "select_from_list_name": "gender_choices"
    },
    {
      "type": "select_one",
      "name": "age_group",
      "label": [
        "\u0645\u0627 \u0647\u0648 \u0641\u0626\u062a\u0643 \u0627\u0644\u0639\u0645\u0631\u064a\u0629\u061f"
      ],
      "required": true,
      "select_from_list_name": "age_group_choices"
    },
    {
      "type": "select_one",
      "name": "governorate",
      "label": [
        "\u0641\u064a \u0623\u064a \u0645\u062d\u0627\u0641\u0638\u0629 \u062a\u0639\u064a\u0634\u061f"
      ],
      "required": true,
      "select_from_list_name": "governorate_choices"
    },
    {
      "type": "select_one",
      "name": "education_level",
      "label": [
        "\u0645\u0627 \u0647\u0648 \u0645\u0633\u062a\u0648\u0627\u0643 \u0627\u0644\u062a\u0639\u0644\u064a\u0645

In [13]:
from Kobo.kobo_form_uploader import KoboFormUploader

uploader = KoboFormUploader(
    api_url='https://kf.kobotoolbox.org/',
    token='f7196ace51297457806144859ca0991c1747c11c'
)

# 1) Create the form
result = uploader.create_form(builder.to_json(), builder.title)
print('Create result:', result)

uid = result.get('uid') or result.get('id')
print('Form UID:', uid)

# 2) Deploy the form
deployment = uploader.deploy_form(uid)
print('Deployed form identifier:', deployment.get('identifier'))


Create result: {'url': 'https://kf.kobotoolbox.org/api/v2/assets/ae78hRrKgdmLMhwgZPRSL5/', 'owner': 'https://kf.kobotoolbox.org/api/v2/users/eyas/', 'owner__username': 'eyas', 'parent': None, 'settings': {'country': [], 'sector': {}, 'description': '', 'organization': '', 'country_codes': []}, 'asset_type': 'survey', 'files': [], 'summary': {'row_count': 10, 'languages': [], 'default_translation': None, 'geo': False, 'lock_all': False, 'lock_any': False, 'labels': ['ما هو جنسك؟', 'ما هو فئتك العمرية؟', 'في أي محافظة تعيش؟', 'ما هو مستواك التعليمي؟', 'ما هو وضعك الوظيفي؟'], 'columns': ['type', 'name', 'label', 'required', 'select_from_list_name'], 'name_quality': {'firsts': {}, 'bad': 0, 'ok': 0, 'good': 10, 'total': 10}}, 'date_created': '2025-07-09T06:00:12.296514Z', 'date_modified': '2025-07-09T06:00:12.318987Z', 'date_deployed': None, 'version_id': 'vFWAS8gT5MymNQHAS3A7UX', 'version__content_hash': 'ddd4757b119c39658e5b99885487af16336e25af', 'version_count': 1, 'has_deployment': Fal

In [6]:
# num_responses = 1000
# responses = []

# for _ in range(num_responses):
#     response = {qid: random.choice(data["options"]) for qid, data in questions.items()}
#     responses.append(response)

# df = pd.DataFrame(responses)

# df.head()


In [7]:
# import featuretools as ft
# import numpy as np
# import pandas as pd

In [8]:
# X = df
# y = df['repaid']


# from sklearn.model_selection import train_test_split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)


# from sklearn.ensemble import RandomForestClassifier
# classifier = RandomForestClassifier(n_estimators=200, random_state=0)  
# classifier.fit(X_train, y_train)  
# predictions = classifier.predict(X_test)


# from sklearn.metrics import classification_report, accuracy_score
# print(classification_report(y_test,predictions ))  
# print(accuracy_score(y_test, predictions ))


# feat_importances = pd.Series(classifier.feature_importances_, index=X.columns)
# feat_importances.nlargest(10).plot(kind='barh')

In [9]:
# I recently came across 'featuretools', which allows users to do automated featue engineering.
# Without knowing a whole lot about the business of 'loans', we could easily make mistakes when doing feature engineering.
# Let's have the computer create some features for us automatically, based on time slices, percentiles, cumulative means, etc., etc., etc.

# import featuretools as ft

# # Create an empty EntitySet
# es = ft.EntitySet(id="customer_data")

# # Add the dataframe to the EntitySet
# es = es.add_dataframe(dataframe_name='client_id', dataframe=df, index='id', make_index=True)

# # Run DFS (Deep Feature Synthesis) to automatically generate features
# feature_matrix, feature_defs = ft.dfs(entityset=es,
#                                       target_dataframe_name="client_id",
#                                       trans_primitives=['time_since', 'day', 'is_weekend',
#                                                         'cum_min', 'minute',
#                                                         'num_words', 'weekday', 'cum_count',
#                                                         'percentile', 'year', 'week',
#                                                         'cum_mean'])

# # List the generated features
# print(list(feature_matrix))


In [10]:
# import pandas as pd
# from featurewiz import featurewiz

# from sklearn import datasets
# import statsmodels.api as sm
# import numpy as np
# import pandas as pd

# mtcars = sm.datasets.get_rdataset("mtcars", "datasets", cache=True).data
# df = pd.DataFrame(mtcars)

# display(df.head())


# X = df.drop(['mpg'],axis=1)
# y = df.mpg.values 


# features = featurewiz(df, target='mpg', corr_limit=0.70, verbose=2)

In [11]:
# from tsfresh import extract_features

# extracted_features = extract_features(df, column_id="client_id", column_sort="client_id")
# print(extracted_features.shape)
# print(extracted_features.columns)